# Training on Kaggle, 2x T4

Notebook settings: accelerator GPU T4 x2, internet on, the competition data and the `hw2_kaggle` dataset (the uncommitted files, laid over the cloned repo) added as input. Run it with Save Version, Save & Run All (Commit), so it keeps running with the browser closed and `/kaggle/working` is saved as the version output.

`run_t4.sh` trains one queue per GPU and stops both after `LIMIT` (10 h 45 min), before Kaggle's 12 h cut. For the next session add the previous version's output as input: its `runs/` is copied back, finished runs are skipped and unfinished ones resume from their last epoch.

For a timing check first, run the training cell with `EXTRA="epochs=1 max_iters=200" OUT=runs_timing` and look at the seconds per epoch in the logs.

In [ ]:
import glob
import os
import shutil
import subprocess
from pathlib import Path

REPO = "https://github.com/llevttarr/ucu-ai-hw-kostiuk-levytskyi.git"
WORK = Path("/kaggle/working/hw2")
if not WORK.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(WORK)], check=True)
os.chdir(WORK)
for extra in glob.glob("/kaggle/input/**/hw2_kaggle/run_t4.sh", recursive=True):
    shutil.copytree(Path(extra).parent, WORK, dirs_exist_ok=True)
    print("copied the uncommitted files from", Path(extra).parent)
subprocess.run(["git", "log", "--oneline", "-1"])

In [ ]:
!pip install -q ultralytics==8.4.189 pycocotools && pip install -q --no-deps -e .

In [ ]:
data = next(Path(p).parent for p in glob.glob("/kaggle/input/**/test_images.csv", recursive=True))
os.environ["HW2_DATA"] = str(data)
os.environ["HW2_YOLO"] = "/tmp/hw2_yolo"
for prev in glob.glob("/kaggle/input/**/hw2/runs", recursive=True):
    shutil.copytree(prev, "runs", dirs_exist_ok=True)
    print("restored", prev)
print(data, sorted(os.listdir(data)))
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!PY=python LIMIT=645m WORKERS=3 bash run_t4.sh

In [ ]:
import pandas as pd

shutil.rmtree("data", ignore_errors=True)
pd.read_csv("runs/experiments.csv") if Path("runs/experiments.csv").exists() else None